# Analyse du dédoublonnage

## Fonction d'analyse
Calcul et restitution du nombre de ligne en défut d'intégrité

In [1]:
from datetime import datetime
import json
from tab_dataset import Cdataset
import pandas as pd
import ntv_pandas as npd
import pathlib

def analyse_integrite(data, schema, affiche=True, indic=True):
    '''analyse les relations du DataFrame 'data' définies dans le schéma 'schema'.
    Le nombre de lignes en erreur par relation (dict) est retourné et optionnellement affiché (paramètre 'affiche=True') . 
    Les lignes en erreur sont optionnellement ajoutées (paramètre 'indic=True') à 'data' sous forme de champs booléens par relation.
    '''
    dic_errors = Cdataset(data).check_relationship(schema)
    dic_count = {name: len(errors) for name, errors in dic_errors.items()}
    if affiche:
        for name, total in dic_count.items():
            print('{:<50} {:>5}'.format(name, total))
    if indic:
        data['ok'] = True
        for name, errors in dic_errors.items():
            data[name] = True
            data.loc[errors, name] = False
            data['ok'] = data['ok'] & data[name] 
        if affiche:
            nb_ok = sum(data['ok'])
            nb_ko = len(data) - sum(data['ok'])      
            print("\nnombre d'enregistrements sans erreurs : ", nb_ok)
            print("nombre d'enregistrements avec au moins une erreur : ", nb_ko)
            print("dont doublons : ", dic_count['index - id_pdc_itinerance'])
            print("\ntaux d'erreur : ", round(nb_ko / len(data) * 100), ' %')
    return dic_count

## Schéma de données
Le schéma de données restreint à la propriété 'relationship' et construit à partir du modèle de données est le suivants :

In [2]:
# complément à inclure dans le schéma de données
schema = {
    'relationships': [
         # relation unicité des pdl
         {"fields": ["id_pdc_itinerance", "index"],                    "link" : "coupled" },   
         # relations inter entités
         {"fields": ["id_station_itinerance", "contact_operateur"],    "link" : "derived" },
         {"fields": ["id_station_itinerance", "nom_enseigne"],         "link" : "derived" },
         {"fields": ["id_station_itinerance", "coordonneesXY"],        "link" : "derived" },
         {"fields": ["id_pdc_itinerance", "id_station_itinerance"],    "link" : "derived" },
         # relations intra entité - station
         {"fields": ["id_station_itinerance", "nom_station"],          "link" : "derived" },
         {"fields": ["id_station_itinerance", "implantation_station"], "link" : "derived" },
         #{"fields": ["id_station_itinerance", "date_maj"],             "link" : "derived" },
         {"fields": ["id_station_itinerance", "nbre_pdc"],             "link" : "derived" },
         {"fields": ["id_station_itinerance", "condition_acces"],      "link" : "derived" },
         {"fields": ["id_station_itinerance", "horaires"],             "link" : "derived" },
         {"fields": ["id_station_itinerance", "station_deux_roues"],   "link" : "derived" },
         # relations intra entité - localisation
         {"fields": ["coordonneesXY", "adresse_station"],              "link" : "derived" }
    ]
}

## Initialisation des données
Fichier pandas

In [3]:
origine = 'datagouv_organization_or_owner'
priorite = 'priorite'
coord = 'coordonneesXY'
id_station = 'id_station_itinerance'
id_pdc = 'id_pdc_itinerance'
last_modif = 'last_modified'
date_maj = 'date_maj'
nom_station = 'nom_station'
adresse = 'adresse_station'
amenageur = 'nom_amenageur'
unit = 'unite'

unicite_stations = [id_station, origine, date_maj, last_modif]
filtre = [priorite,  date_maj, last_modif]
id_station_pdc = [id_station, id_pdc]
att_station = [id_station, date_maj, last_modif, amenageur, nom_station, coord]
att_pdc = [id_pdc, id_station, date_maj, last_modif, amenageur, nom_station, coord, adresse, origine]
filtre_qualicharge = [nom_station, adresse, coord]

Consolidation statique dédoublonnée : https://proxy.transport.data.gouv.fr/resource/consolidation-transport-irve-statique
Consolidation avec doublons : https://proxy.transport.data.gouv.fr/resource/consolidation-transport-avec-doublons-irve-statique

In [4]:
file_irve_brut = 'consolidation_transport_avec_doublons_irve_statique_14_03.csv' # données brutes
file_irve = 'consolidation_transport_irve_statique_14_03.csv' # données dédoublonnées

irve_brut = pd.read_csv(file_irve_brut, sep=',', low_memory=False, dtype='object').reset_index()
irve_brut[last_modif] = irve_brut['datagouv_last_modified']

irve = pd.read_csv(file_irve, sep=',', low_memory=False, dtype='object').reset_index()
irve[last_modif] = irve['datagouv_last_modified']

In [81]:
file_datagouv = 'consolidation-etalab-schema-irve-statique-v-2.3.1-20260314.csv'
irve_brut = pd.read_csv(file_datagouv, sep=',', low_memory=False, dtype='object').reset_index()

## Données brutes

In [82]:
print('nombre de lignes : {}, nombre de pdc : {} \n'.format(len(irve_brut), len(irve_brut.groupby([id_pdc]).count())))
print(irve_brut.groupby([origine]).count()['index'].sort_values(ascending=False)[0:10], '\n')
res_brut = analyse_integrite(irve_brut, schema)

nombre de lignes : 213792, nombre de pdc : 152468 

datagouv_organization_or_owner
qualicharge                       61079
gireve-2                          32363
izivia                            14717
alize                             14449
eco-movement                      11996
indigo-group                       7273
power-dot-france                   6611
totalenergies-marketing-france     6282
lidl                               5562
driveco                            5065
Name: index, dtype: int64 

index - id_pdc_itinerance                          114581
contact_operateur - id_station_itinerance          34721
nom_enseigne - id_station_itinerance               42022
coordonneesXY - id_station_itinerance              52220
id_station_itinerance - id_pdc_itinerance          56857
nom_station - id_station_itinerance                20737
implantation_station - id_station_itinerance       25558
nbre_pdc - id_station_itinerance                   26913
condition_acces - id_station_iti

## Test des cas de doublons identifiés

- cas 0 : doublon de pdc d'origine différentes sur une même station
- cas 1 : pdc sur deux stations avec identifiants de station différents
- cas 2 : station avec deux origines, des identifiants différents et mêmes coordonnées
- cas 3 : station avec deux origines, des identifiants et des coordonnées différents, des noms identiques
- cas 4 : station avec deux origines, des identifiants et des coordonnées et des noms différents, des adresses identiques
- cas 5 : station Qualicharge décommissionnée partiellement
- cas 6 : station Qualicharge décommissionnée totalement
- cas 7 : station Qualicharge avec changement d'unité d'exploitation

In [6]:
def test_doublons(irve):
    return {
        'station Tesla de 48 pdc (cas 0)': len(irve[irve[id_station]=='FRTSLP16281'])==48,
        'pdc Atlante sur deux stations (cas 1)': len(irve[irve[id_pdc]=='FRATLE102181'])==1,
        'pdc Ionity sur deux stations (cas 1)': len(irve[irve[id_pdc]=='FRIOYE423408']) == 1,                 
        'stations de 2 pdc (cas 2)': len(irve[irve[coord]=='[-0.00097, 49.32456]'])==2,
        'stations de 3 pdc (cas 2)': len(irve[irve[coord]=='[-0.03405, 48.75304]'])==3,
        'stations de 12 pdc (cas 2)': len(irve[irve[coord]=='[-0.05677, 48.72293]'])==12,
        'stations de 2 pdc (cas 3)': len(irve[irve[nom_station]=='VALENCE EN POITOU_SALLE DES FETES'])==2,
        'stations de 3 pdc (cas 4)': len(irve[irve[adresse]=='15 Av. Président Georges Pompidou'])==3,
        'station R3 de 1 pdc (cas 5)': len(irve[irve[id_station]=='FRR3MP1063577'])==1,
        'station R3 de 3 pdc (cas 6)': len(irve[irve[id_station]=='FRR3MP1030133'])==1,
        'station Total (cas 7)': len(irve[irve[coord]=='[5.40266, 43.26239]'])==2
    }
def cumul_tests(resultat):
    return sum(resultat.values())

## Dédoublonnage actuel

In [7]:
print('nombre de lignes : {} \n'.format(len(irve)))
print(irve.groupby([origine]).count()['index'].sort_values(ascending=False)[0:10], '\n')
print("nombre d'erreur d'intégrité par type de règle :\n")
res_actuel = analyse_integrite(irve, schema)
test_actuel = test_doublons(irve)
print("\nbilan des tests :\n")
test_actuel

nombre de lignes : 149777 

datagouv_organization_or_owner
QualiCharge                                  61079
GIREVE                                       26135
Alizé                                        13847
IZIVIA                                       13009
Indigo Group                                  7273
Driveco                                       3919
Eco-Movement                                  3168
TotalEnergies Marketing France                1849
Engie Mobilités Electriques                   1729
Citeos Ingénierie IdF & Est (Cogelum IdF)     1498
Name: index, dtype: int64 

nombre d'erreur d'intégrité par type de règle :

index - id_pdc_itinerance                              0
contact_operateur - id_station_itinerance           1823
nom_enseigne - id_station_itinerance                4103
coordonneesXY - id_station_itinerance               5407
id_station_itinerance - id_pdc_itinerance              0
nom_station - id_station_itinerance                 2351
implantation

{'station Tesla de 48 pdc (cas 0)': True,
 'pdc Atlante sur deux stations (cas 1)': True,
 'pdc Ionity sur deux stations (cas 1)': True,
 'stations de 2 pdc (cas 2)': False,
 'stations de 3 pdc (cas 2)': False,
 'stations de 12 pdc (cas 2)': False,
 'stations de 2 pdc (cas 3)': False,
 'stations de 3 pdc (cas 4)': False,
 'station R3 de 1 pdc (cas 5)': False,
 'station R3 de 3 pdc (cas 6)': False,
 'station Total (cas 7)': False}

## Dédoublonnage proposé
Les critères utilisés pour éliminer les doublons sont par ordre : priorite (datagouv_organization_or_owner), date (date_maj, datagouv_last_modified)

In [83]:
# dédoublonnage des stations (suivant critères d'unicité)
stations = irve_brut.drop_duplicates(unicite_stations).copy()

print("nombre de stations brutes {}, stations suivant critère d'unicité {} et stations uniques {} ".format(len(irve_brut), len(stations), len(irve_brut.drop_duplicates(id_station))))

nombre de stations brutes 213792, stations suivant critère d'unicité 71893 et stations uniques 61395 


In [97]:
stations.groupby([origine]).count()['index'].sort_values(ascending=False)[0:15]

datagouv_organization_or_owner
qualicharge                                13684
eco-movement                               10684
gireve-2                                   10373
izivia                                      7500
alize                                       4528
citeos-ingenierie-idf-est-cogelum-idf       2279
spbr1                                       2117
greenea                                     1910
load-stations                               1368
driveco                                     1311
lidl                                        1058
power-dot-france                            1053
totalenergies-marketing-france               967
syndicat-departemental-denergie-du-tarn      700
ze-watt                                      629
Name: index, dtype: int64

### Dédoublonnage direct station
A l'issue de cette étape, on a une liste de stations uniques respectant les critères de filtrage

avec l'origine la plus prioritaire et la mise à jour la plus récente

In [108]:
# choix du critère de priorité (Qualicharge)
stations[priorite] = stations[origine].str.lower() == 'qualicharge'

# dédoublonnage des stations suivant son id_station_itinerance avec filtrage suivant les critères retenus
stat_direct = stations.sort_values(by=[id_station] + filtre).drop_duplicates(id_station, keep='last').copy()
if stations[priorite].sum() != stat_direct[priorite].sum():
    print('priorité non respectée')
print('Stations Qualicharge : {}'.format(stations[priorite].sum()))

Stations Qualicharge : 13684


In [109]:
print('nombre de stations initiales {}, stations dédoublonnées {} (supprimées {})'.format(len(stations), len(stat_direct), len(stations) - len(stat_direct)))
stat_direct.groupby([origine]).count()['index'].sort_values(ascending=False)[0:15]

nombre de stations initiales 71893, stations dédoublonnées 61395 (supprimées 10498)


datagouv_organization_or_owner
qualicharge                                13684
eco-movement                               10684
gireve-2                                   10069
izivia                                      6937
alize                                       4528
citeos-ingenierie-idf-est-cogelum-idf       1728
greenea                                     1692
load-stations                               1368
driveco                                      814
syndicat-departemental-denergie-du-tarn      700
ze-watt                                      623
soregies                                     451
plus-de-bornes                               394
zeborne                                      377
mobilize-power-solutions                     355
Name: index, dtype: int64

### Dédoublonnage direct pdc
A l'issue de cette étape, chaque pdc est présent une seule fois sur la station respectant les critères de filtrage

In [110]:
# dédoublonnage des pdc présents sur plusieurs stations
pdc_stat = stat_direct[unicite_stations + [priorite]].merge(irve_brut, how='left', on=unicite_stations)
pdc_stat_unique = pdc_stat.sort_values(by=id_station_pdc + filtre).drop_duplicates(id_station_pdc, keep='last').copy()

# dédoublonnage des pdc suivant son id_pdc avec filtrage par priorite, date_maj, last_modif (priorité déja filtrée sur les stations mais nécessaire)
pdc_direct =  pdc_stat_unique.sort_values(by=[id_pdc] + filtre).drop_duplicates(id_pdc, keep='last').copy()
del(pdc_direct['index'])
pdc_direct = pdc_direct.reset_index()

### Bilan dédoublonnage direct

In [111]:
print("nombre de pdc initial {}, avec dédoublonnage des pdc multi-stations {} et avec dédoublonnage de l'historique {} (supprimés {})\n".format(len(pdc_stat), len(pdc_stat_unique), len(pdc_direct), len(pdc_stat)-len(pdc_direct)))
print(pdc_direct.groupby([origine]).count()['index'].sort_values(ascending=False)[0:10], '\n')
print("nombre d'erreur d'intégrité par type de règle :\n")
res_direct = analyse_integrite(pdc_direct, schema)
propos_direct = test_doublons(pdc_direct)
print("\nbilan des tests :\n")
propos_direct

nombre de pdc initial 175842, avec dédoublonnage des pdc multi-stations 175842 et avec dédoublonnage de l'historique 149056 (supprimés 26786)

datagouv_organization_or_owner
qualicharge                              61079
gireve-2                                 23738
alize                                    13857
izivia                                   12636
indigo-group                              7273
eco-movement                              4145
citeos-ingenierie-idf-est-cogelum-idf     3278
driveco                                   3070
totalenergies-marketing-france            1848
engie-mobilites-electriques               1747
Name: index, dtype: int64 

nombre d'erreur d'intégrité par type de règle :

index - id_pdc_itinerance                              0
contact_operateur - id_station_itinerance              2
nom_enseigne - id_station_itinerance                  14
coordonneesXY - id_station_itinerance               1019
id_station_itinerance - id_pdc_itinerance          

{'station Tesla de 48 pdc (cas 0)': True,
 'pdc Atlante sur deux stations (cas 1)': True,
 'pdc Ionity sur deux stations (cas 1)': True,
 'stations de 2 pdc (cas 2)': False,
 'stations de 3 pdc (cas 2)': False,
 'stations de 12 pdc (cas 2)': False,
 'stations de 2 pdc (cas 3)': False,
 'stations de 3 pdc (cas 4)': False,
 'station R3 de 1 pdc (cas 5)': True,
 'station R3 de 3 pdc (cas 6)': False,
 'station Total (cas 7)': False}

### Dédoublonnage indirect des stations

In [112]:
stations_direct = pdc_direct.drop_duplicates([id_station]).copy()

In [113]:
# Cette fonction donne la liste des stations en éliminant (suivant les critères de filtrage) celles avec le champ 'attribut' identique et venant de plusieurs origines
def indirect_station_ext(stations, attribut, affiche=True):
    dupl_att_origine = ~stations.duplicated(keep=False, subset=[attribut, origine])
    
    stations_ext = stations[dupl_att_origine].copy()
    stations_int = stations[~dupl_att_origine].copy()

    filtrage = stations_ext.sort_values(by=[attribut, priorite])
    stat_att = filtrage.drop_duplicates([attribut], keep='last').copy()
    
    duplicates = filtrage.duplicated(subset=[attribut], keep='last')
    #print(duplicates)
    duplicated = stations_ext.loc[duplicates] #.copy()
    
    resultat = pd.concat([stations_int, stat_att])
    if affiche :
        print("nombre de stations dupliquées pour l'attribut '{:<15}' : {}".format(attribut, len(stations_ext) - len(stat_att)))
        print("nombre initial de stations {}, avec dédoublonnage {} {}\n".format(len(stations), attribut, len(resultat)))
    return (resultat, duplicated)

#### Evaluation des stations avec un attribut identique et origine différente
Cette étape donne la liste des stations en éliminant (suivant les critères de filtrage) celles avec un attribut identiques et venant de plusieurs origines

In [114]:
stations_xy, dupl_xy = indirect_station_ext(stations_direct, coord)
stations_nom, dupl_nom = indirect_station_ext(stations_direct, nom_station)
stations_adr, dupl_adr = indirect_station_ext(stations_direct, adresse)

nombre de stations dupliquées pour l'attribut 'coordonneesXY  ' : 126
nombre initial de stations 47276, avec dédoublonnage coordonneesXY 47150

nombre de stations dupliquées pour l'attribut 'nom_station    ' : 197
nombre initial de stations 47276, avec dédoublonnage nom_station 47079

nombre de stations dupliquées pour l'attribut 'adresse_station' : 107
nombre initial de stations 47276, avec dédoublonnage adresse_station 47169



#### Dédoublonnage des stations hors Qualicharge

In [115]:
stations_xy, dupl_xy = indirect_station_ext(stations_direct, coord)

nombre de stations dupliquées pour l'attribut 'coordonneesXY  ' : 126
nombre initial de stations 47276, avec dédoublonnage coordonneesXY 47150



In [116]:
stations_nom, dupl_nom = indirect_station_ext(stations_xy, nom_station)

nombre de stations dupliquées pour l'attribut 'nom_station    ' : 133
nombre initial de stations 47150, avec dédoublonnage nom_station 47017



In [117]:
stations_adr, dupl_adr = indirect_station_ext(stations_nom, adresse)

nombre de stations dupliquées pour l'attribut 'adresse_station' : 37
nombre initial de stations 47017, avec dédoublonnage adresse_station 46980



In [118]:
# le gain avec la suppression des doublons d'adresse est faible et on supprime quelques stations différentes, on se limite aux coordonnées et au nom
stations_indirect = stations_nom

In [119]:
stations_indirect.groupby([origine]).count()['index'].sort_values(ascending=False)[0:15]

datagouv_organization_or_owner
qualicharge                                13684
gireve-2                                    7268
izivia                                      6936
alize                                       4245
eco-movement                                4058
citeos-ingenierie-idf-est-cogelum-idf       1717
driveco                                      808
ze-watt                                      623
syndicat-departemental-denergie-du-tarn      600
soregies                                     404
plus-de-bornes                               394
zeborne                                      377
mobilize-power-solutions                     355
rossini-energy                               311
engie-mobilites-electriques                  310
Name: index, dtype: int64

#### Dédoublonnage des stations Qualicharge
origine Qualicharge, même nom, même coordonnées, même adresse, unité différente

In [120]:
# Cette fonction donne la liste des stations en éliminant (suivant les critères de filtrage) les stations Qualicharge avec les mêmes champs nom, coordonnées et adresse mais des unités différentes
def indirect_qualicharge(stations, affiche=True):
    stat = stations.copy()
    stat[unit] = stat[id_station].str[:5]
    #filtre_qualicharge = [nom_station, adresse, coord]
    
    stat_quali = stat[stat[origine]=='QualiCharge'].copy()
    stat_not_quali = stat[~(stat[origine]=='QualiCharge')].copy()
    
    stat_quali['unique'] = ~stat_quali.duplicated(filtre_qualicharge + [unit])
    filtrage = stat_quali.sort_values(by=['unique'] + filtre_qualicharge + [date_maj])
    stat_quali['doublon'] = filtrage.duplicated(subset=filtre_qualicharge, keep='last')
    
    duplicated = stat_quali[stat_quali['doublon']]
    stat_result = stat_quali[~stat_quali['doublon']]
    stat_result.drop(['doublon', 'unique'], axis=1)

    resultat = pd.concat([stat_not_quali, stat_result])
    resultat.drop(['unite'], axis=1)
    
    if affiche :
        print("nombre de stations dupliquées entre unités : {}".format(len(duplicated)))
        print("nombre initial de stations {}, avec dédoublonnage {}\n".format(len(stations), len(resultat)))
    return (resultat, duplicated)

In [121]:
stat_indirect_quali, dupl_quali = indirect_qualicharge(stations_indirect)

nombre de stations dupliquées entre unités : 0
nombre initial de stations 47017, avec dédoublonnage 47017



### Dédoublonnage indirect pdc

In [122]:
pdc_indirect = stat_indirect_quali[[id_station]].merge(pdc_direct, how='left', on=id_station)

### Bilan dédoublonnage indirect

In [123]:
print("nombre de pdc initial {} et avec dédoublonnage {} (supprimés {})\n".format(len(pdc_direct), len(pdc_indirect), len(pdc_direct)-len(pdc_indirect)))
print(pdc_indirect.groupby([origine]).count()['index'].sort_values(ascending=False)[0:15], '\n')
print("nombre d'erreur d'intégrité par type de règle :\n")
res_indirect = analyse_integrite(pdc_indirect, schema)
propos_indirect = test_doublons(pdc_indirect)
print("\nbilan des tests :\n")
propos_indirect

nombre de pdc initial 149056 et avec dédoublonnage 148677 (supprimés 379)

datagouv_organization_or_owner
qualicharge                              61079
gireve-2                                 23738
alize                                    13837
izivia                                   12632
indigo-group                              7273
eco-movement                              4145
citeos-ingenierie-idf-est-cogelum-idf     3254
driveco                                   3070
totalenergies-marketing-france            1845
engie-mobilites-electriques               1745
plus-de-bornes                            1300
qovoltis                                  1198
lidl                                      1077
e-totem                                    896
rossini-energy                             855
Name: index, dtype: int64 

nombre d'erreur d'intégrité par type de règle :

index - id_pdc_itinerance                              0
contact_operateur - id_station_itinerance              

{'station Tesla de 48 pdc (cas 0)': True,
 'pdc Atlante sur deux stations (cas 1)': True,
 'pdc Ionity sur deux stations (cas 1)': True,
 'stations de 2 pdc (cas 2)': False,
 'stations de 3 pdc (cas 2)': False,
 'stations de 12 pdc (cas 2)': False,
 'stations de 2 pdc (cas 3)': True,
 'stations de 3 pdc (cas 4)': False,
 'station R3 de 1 pdc (cas 5)': True,
 'station R3 de 3 pdc (cas 6)': False,
 'station Total (cas 7)': False}

## Synthèse des dédoublonnages

In [124]:
print("données brutes       : total pdc {:<7} (nombre d'id_pdc_itinerance différents)".format(len(irve_brut.groupby([id_pdc]).count())))
print('solution actuelle    : total pdc {:<7} avec pdc ok {:<7}, pdc avec erreur {:>5} dont doublons {}, nombre de tests réussis {}'.format(len(irve), sum(irve['ok']), len(irve) - sum(irve['ok']), res_actuel['index - id_pdc_itinerance'], sum(test_actuel.values())))
print('proposition direct   : total pdc {:<7} avec pdc ok {:<7}, pdc avec erreur {:>5} dont doublons {}, nombre de tests réussis {}'.format(len(pdc_direct), sum(pdc_direct['ok']), len(pdc_direct) - sum(pdc_direct['ok']), res_direct['index - id_pdc_itinerance'], sum(propos_direct.values())))
print('proposition indirect : total pdc {:<7} avec pdc ok {:<7}, pdc avec erreur {:>5} dont doublons {}, nombre de tests réussis {}'.format(len(pdc_indirect), sum(pdc_indirect['ok']), len(pdc_indirect) - sum(pdc_indirect['ok']), res_indirect['index - id_pdc_itinerance'], sum(propos_indirect.values())))


données brutes       : total pdc 152468  (nombre d'id_pdc_itinerance différents)
solution actuelle    : total pdc 149777  avec pdc ok 134731 , pdc avec erreur 15046 dont doublons 0, nombre de tests réussis 3
proposition direct   : total pdc 149056  avec pdc ok 144353 , pdc avec erreur  4703 dont doublons 0, nombre de tests réussis 4
proposition indirect : total pdc 148677  avec pdc ok 144749 , pdc avec erreur  3928 dont doublons 0, nombre de tests réussis 5


## Exemples

In [125]:
irve[irve[coord]=='[-0.03405, 48.75304]'][att_pdc]

,id_pdc_itinerance,id_station_itinerance,date_maj,last_modified,nom_amenageur,nom_station,coordonneesXY,adresse_station,datagouv_organization_or_owner
34772,FREVZEKIEU1,FREVZP4070785939483057665,2025-09-24,2026-02-01T03:01:17.000000+0000,SMEG Développement,EVzen/3B222B60-EC8D-4077-AB1F-1C6D169CB4B1,"[-0.03405, 48.75304]","Rue Maurice Ravel, Argentan 61200 France",GIREVE
34773,FREVZELOMA1,FREVZP4070785939483057665,2025-09-24,2026-02-01T03:01:17.000000+0000,SMEG Développement,EVzen/3B222B60-EC8D-4077-AB1F-1C6D169CB4B1,"[-0.03405, 48.75304]","Rue Maurice Ravel, Argentan 61200 France",GIREVE
34774,FREVZEKIEU2,FREVZP4070785939483057665,2025-09-24,2026-02-01T03:01:17.000000+0000,SMEG Développement,EVzen/3B222B60-EC8D-4077-AB1F-1C6D169CB4B1,"[-0.03405, 48.75304]","Rue Maurice Ravel, Argentan 61200 France",GIREVE
85547,FREVZEFREVZELOMA1,FREVZP3B222B60EC8D4077AB1F1C6D169CB,2026-03-13,2026-03-13T23:59:49.000000+0000,SMEG DEVELOPPEMENT,"EVzen - Argentan, PRIX MIAM Argentan","[-0.03405, 48.75304]",Rue Maurice Ravel 61200 Argentan,QualiCharge
99307,FREVZEFREVZEKIEU1,FREVZP3B222B60EC8D4077AB1F1C6D169CB,2026-03-13,2026-03-13T23:59:49.000000+0000,SMEG DEVELOPPEMENT,"EVzen - Argentan, PRIX MIAM Argentan","[-0.03405, 48.75304]",Rue Maurice Ravel 61200 Argentan,QualiCharge
99360,FREVZEFREVZEKIEU2,FREVZP3B222B60EC8D4077AB1F1C6D169CB,2026-03-13,2026-03-13T23:59:49.000000+0000,SMEG DEVELOPPEMENT,"EVzen - Argentan, PRIX MIAM Argentan","[-0.03405, 48.75304]",Rue Maurice Ravel 61200 Argentan,QualiCharge


In [126]:
irve[irve[coord]=='[5.40266, 43.26239]'][att_pdc]

,id_pdc_itinerance,id_station_itinerance,date_maj,last_modified,nom_amenageur,nom_station,coordonneesXY,adresse_station,datagouv_organization_or_owner
131793,FRTCBE008442,FRTCBP01910,2025-12-21,2026-03-13T23:59:49.000000+0000,TotalEnergies Charging Services,AMP | 1 Rue Mignard,"[5.40266, 43.26239]","1 Rue Mignard, 13009 MARSEILLE",QualiCharge
132023,FRTCBE008441,FRTCBP01910,2025-12-21,2026-03-13T23:59:49.000000+0000,TotalEnergies Charging Services,AMP | 1 Rue Mignard,"[5.40266, 43.26239]","1 Rue Mignard, 13009 MARSEILLE",QualiCharge
143609,FRHXWE008441,FRHXWP01910,2026-01-07,2026-03-13T23:59:49.000000+0000,TotalEnergies Charging Services,AMP | 1 Rue Mignard,"[5.40266, 43.26239]","1 Rue Mignard, 13009 MARSEILLE",QualiCharge
145445,FRHXWE008442,FRHXWP01910,2026-01-07,2026-03-13T23:59:49.000000+0000,TotalEnergies Charging Services,AMP | 1 Rue Mignard,"[5.40266, 43.26239]","1 Rue Mignard, 13009 MARSEILLE",QualiCharge


In [127]:
irve_brut[irve_brut[id_pdc] == 'FRIOYE423402'][att_pdc]

,id_pdc_itinerance,id_station_itinerance,date_maj,last_modified,nom_amenageur,nom_station,coordonneesXY,adresse_station,datagouv_organization_or_owner
83150,FRIOYE423402,FRIOYE423402,2026-03-13,2026-03-13T20:00:21.587000+00:00,IONITY,IONITY Sorgues,"[4.88965220,44.02444820]","Aire de Sorgues, A7, 84700 Sorgues",eco-movement
83151,FRIOYE423402,FRIOYP9587247,2026-03-12,2026-03-13T23:59:49.689736+00:00,Ionity,IONITY Sorgues,"[4.88964, 44.024414]","Aire de Sorgues, A7, 84700 Sorgues",qualicharge


In [128]:
irve_brut[irve_brut[id_station] == 'FRR3MP1030133'][att_pdc]

,id_pdc_itinerance,id_station_itinerance,date_maj,last_modified,nom_amenageur,nom_station,coordonneesXY,adresse_station,datagouv_organization_or_owner
148234,FRR3ME5284264,FRR3MP1030133,2026-01-07,2026-01-08T05:09:01.686000+00:00,R3,Aulnoy-Lez-Valenciennes - NORAUTO,"[3.511244, 50.3332]","Rue Jules Mousseron, 59300 Aulnoy-lez-Valencie...",r3
148235,FRR3ME5284266,FRR3MP1030133,2026-01-07,2026-01-08T05:09:01.686000+00:00,R3,Aulnoy-Lez-Valenciennes - NORAUTO,"[3.511244, 50.3332]","Rue Jules Mousseron, 59300 Aulnoy-lez-Valencie...",r3
148236,FRR3ME5284267,FRR3MP1030133,2026-01-07,2026-01-08T05:09:01.686000+00:00,R3,Aulnoy-Lez-Valenciennes - NORAUTO,"[3.511244, 50.3332]","Rue Jules Mousseron, 59300 Aulnoy-lez-Valencie...",r3


In [129]:
irve[irve[id_station]=='FRATLP1136899124034716498'][att_pdc]

,id_pdc_itinerance,id_station_itinerance,date_maj,last_modified,nom_amenageur,nom_station,coordonneesXY,adresse_station,datagouv_organization_or_owner


In [130]:
irve_brut[irve_brut[id_station]=='FRATLPFR01092'][att_pdc]

,id_pdc_itinerance,id_station_itinerance,date_maj,last_modified,nom_amenageur,nom_station,coordonneesXY,adresse_station,datagouv_organization_or_owner
14989,FRATLE102171,FRATLPFR01092,2026-03-14,2026-03-13T23:59:49.689736+00:00,Atlante,Atlante - Castelnaudary - Intermarché HYPER Ca...,"[1.95725, 43.2998]","Route de Villasavary , 211 , Castelnaudary",qualicharge
14991,FRATLE102172,FRATLPFR01092,2026-03-14,2026-03-13T23:59:49.689736+00:00,Atlante,Atlante - Castelnaudary - Intermarché HYPER Ca...,"[1.95725, 43.2998]","Route de Villasavary , 211 , Castelnaudary",qualicharge
14993,FRATLE102181,FRATLPFR01092,2026-03-14,2026-03-13T23:59:49.689736+00:00,Atlante,Atlante - Castelnaudary - Intermarché HYPER Ca...,"[1.95725, 43.2998]","Route de Villasavary , 211 , Castelnaudary",qualicharge
14995,FRATLE102182,FRATLPFR01092,2026-03-14,2026-03-13T23:59:49.689736+00:00,Atlante,Atlante - Castelnaudary - Intermarché HYPER Ca...,"[1.95725, 43.2998]","Route de Villasavary , 211 , Castelnaudary",qualicharge
14997,FRATLE102191,FRATLPFR01092,2026-03-14,2026-03-13T23:59:49.689736+00:00,Atlante,Atlante - Castelnaudary - Intermarché HYPER Ca...,"[1.95725, 43.2998]","Route de Villasavary , 211 , Castelnaudary",qualicharge
14999,FRATLE102192,FRATLPFR01092,2026-03-14,2026-03-13T23:59:49.689736+00:00,Atlante,Atlante - Castelnaudary - Intermarché HYPER Ca...,"[1.95725, 43.2998]","Route de Villasavary , 211 , Castelnaudary",qualicharge
15001,FRATLE102201,FRATLPFR01092,2026-03-14,2026-03-13T23:59:49.689736+00:00,Atlante,Atlante - Castelnaudary - Intermarché HYPER Ca...,"[1.95725, 43.2998]","Route de Villasavary , 211 , Castelnaudary",qualicharge
15003,FRATLE102202,FRATLPFR01092,2026-03-14,2026-03-13T23:59:49.689736+00:00,Atlante,Atlante - Castelnaudary - Intermarché HYPER Ca...,"[1.95725, 43.2998]","Route de Villasavary , 211 , Castelnaudary",qualicharge
15297,FRATLE104381,FRATLPFR01092,2026-03-14,2026-03-13T23:59:49.689736+00:00,Atlante,Atlante - Castelnaudary - Intermarché HYPER Ca...,"[1.95725, 43.2998]","Route de Villasavary , 211 , Castelnaudary",qualicharge
15299,FRATLE104382,FRATLPFR01092,2026-03-14,2026-03-13T23:59:49.689736+00:00,Atlante,Atlante - Castelnaudary - Intermarché HYPER Ca...,"[1.95725, 43.2998]","Route de Villasavary , 211 , Castelnaudary",qualicharge


In [131]:
irve_brut[irve_brut[adresse]=='15 Av. Président Georges Pompidou'][att_pdc]

,id_pdc_itinerance,id_station_itinerance,date_maj,last_modified,nom_amenageur,nom_station,coordonneesXY,adresse_station,datagouv_organization_or_owner
70715,FRGSPE10000560051,FRGSPP89228259,2026-03-13,2026-03-13T23:59:49.689736+00:00,Greenspot,Carrefour Market Figeac,"[2.024047, 44.606503]",15 Av. Président Georges Pompidou,qualicharge
70716,FRGSPE10000560052,FRGSPP89228259,2026-03-13,2026-03-13T23:59:49.689736+00:00,Greenspot,Carrefour Market Figeac,"[2.024047, 44.606503]",15 Av. Président Georges Pompidou,qualicharge
70717,FRGSPE10000560053,FRGSPP89228259,2026-03-13,2026-03-13T23:59:49.689736+00:00,Greenspot,Carrefour Market Figeac,"[2.024047, 44.606503]",15 Av. Président Georges Pompidou,qualicharge
71398,FRGSPP10000560051,FRGSPP1000056005,2024-06-03,2024-08-01T08:09:38.139000+00:00,CARREFOUR FIGEAC,CARREFOUR FIGEAC,"[2.024519466862065,44.60613271767822]",15 Av. Président Georges Pompidou,greenspot-enersoft
71399,FRGSPP10000560052,FRGSPP1000056005,2024-06-03,2024-08-01T08:09:38.139000+00:00,CARREFOUR FIGEAC,CARREFOUR FIGEAC,"[2.024519466862065,44.60613271767822]",15 Av. Président Georges Pompidou,greenspot-enersoft


In [132]:
irve[irve[id_pdc]=='FRATLE102181'][att_pdc]

,id_pdc_itinerance,id_station_itinerance,date_maj,last_modified,nom_amenageur,nom_station,coordonneesXY,adresse_station,datagouv_organization_or_owner
118905,FRATLE102181,FRATLPFR01092,2026-03-14,2026-03-13T23:59:49.000000+0000,Atlante,Atlante - Castelnaudary - Intermarché HYPER Ca...,"[1.95725, 43.29980]","Route de Villasavary , 211 , Castelnaudary",QualiCharge


In [135]:
irve_brut[irve_brut[coord]=='[-0.00097, 49.32456]'][att_pdc]

,id_pdc_itinerance,id_station_itinerance,date_maj,last_modified,nom_amenageur,nom_station,coordonneesXY,adresse_station,datagouv_organization_or_owner


In [136]:
irve[irve[coord]=='[-0.00097, 49.32456]'][att_pdc]

,id_pdc_itinerance,id_station_itinerance,date_maj,last_modified,nom_amenageur,nom_station,coordonneesXY,adresse_station,datagouv_organization_or_owner
30968,FRS14ETAO20592,FRS14P7162008418302887905,2026-01-28,2026-02-01T03:01:17.000000+0000,Syndicat Mixte Départemental d'Énergies du Cal...,MobiSDEC/69135fc5e3675fc7d3fc2223,"[-0.00097, 49.32456]","Place du Lieutenant Fernand Fanneau, Villers-...",GIREVE
30969,FRS14ETAO20591,FRS14P7162008418302887905,2026-01-28,2026-02-01T03:01:17.000000+0000,Syndicat Mixte Départemental d'Énergies du Cal...,MobiSDEC/69135fc5e3675fc7d3fc2223,"[-0.00097, 49.32456]","Place du Lieutenant Fernand Fanneau, Villers-...",GIREVE
109226,FRS14ERHZJ2,FRS14PLLXJ0T1CUX9Q4H,2026-01-15,2026-03-13T23:59:49.000000+0000,"SM DEPARTEMENTAL D'ENERGIES DU CALVADOS DIT ""S...","Villers-sur-Mer, Place du Casino","[-0.00097, 49.32456]","Villers-sur-Mer, Place du Casino, 14640 Viller...",QualiCharge
109302,FRS14ERHZJ1,FRS14PLLXJ0T1CUX9Q4H,2026-01-15,2026-03-13T23:59:49.000000+0000,"SM DEPARTEMENTAL D'ENERGIES DU CALVADOS DIT ""S...","Villers-sur-Mer, Place du Casino","[-0.00097, 49.32456]","Villers-sur-Mer, Place du Casino, 14640 Viller...",QualiCharge
